LangChain Exercises
=====


In [53]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [2]:
project_root = Path().cwd().parent

load_dotenv(project_root / ".env")
assert os.getenv("GROQ_API_KEY"), "Please put the API key in '.env'"

In [10]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="hidden"
)

conversation = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("What is the closest plant to Earth on average?"),
    AIMessage("Mercury is the closest planet to Earth on average."),
    HumanMessage("Which planet has the minium distance to Earth?")
]

response = llm.invoke(conversation)
console = Console()
md = Markdown(response.content)
console.print(md)


The planet with the minimum distance to Earth is Venus.                                                            

Key Details:                                                                                                       

 • Minimum distance: When Earth and Venus are at their closest approach (i.e., when Venus is at inferior           
   conjunction), the distance can be as little as 38 million kilometers (24 million miles).                        
 • Comparison:                                                                                                     
    • Mercury (closest planet to the Sun) has a minimum distance of about 77 million kilometers (48 million miles) 
      to Earth.                                                                                                    
    • Mars has a minimum distance of about 54.6 million kilometers (34 million miles) when aligned with Earth.     

Why the Difference?                                                                                                

 • Average distance vs. minimum distance:                                                                          
    • Mercury is the closest planet to Earth on average (due to its orbit being closer to the Sun), but Venus is   
      the nearest neighbor in terms of minimum distance when their orbits align.                                   
    • This distinction arises because average distance considers the entire orbital paths, while minimum distance  
      refers to the closest possible approach.                                                                     

So, while Mercury is the "closest" in the long-term average sense, Venus is the planet that physically comes       
closest to Earth at specific points in time. 🌍🪐

### Streaming

In [18]:
for chunk in llm.stream("Hi"):
    print(chunk.content)

for chunk in llm.stream(conversation):
    print(chunk.text, end="", flush=True)


Hello
!
 How
 can
 I
 assist
 you
 today
?
 😊


The planet with the **minimum distance** to Earth is **Venus**. 

### Key Details:
- **Venus** can come as close as **~38 million kilometers** to Earth during its closest approach (inferior conjunction), when it is directly between Earth and the Sun. 
- **Mercury**, while the **closest planet on average** (due to its proximity to the Sun), only reaches a minimum distance of **~77 million kilometers** from Earth. 
- **Mars** can also approach Earth closely (~55 million km at opposition), but not as closely as Venus.

### Why the Difference?
- **Average distance** considers the long-term orbital positions of planets. Mercury’s orbit is closest to the Sun, so it’s statistically the "closest" planet on average.
- **Minimum distance** refers to the closest possible approach during specific orbital alignments. Venus, with its smaller orbit than Earth, achieves this closer proximity.

So, **Venus** is the correct answer for the **minimum distan

### Weather

In [30]:
import requests
from langchain.tools import tool, ToolRuntime


@tool
def get_weather(city: str):
    """A tool to get wetaher for a given city"""
    response = requests.get(f"https://wttr.in/{city}?format=j1")
    return response.json()

get_weather.invoke({"city": "tokyo"})

{'current_condition': [{'FeelsLikeC': '16',
   'FeelsLikeF': '61',
   'cloudcover': '83',
   'humidity': '88',
   'observation_time': '07:03 PM',
   'precipInches': '0.0',
   'precipMM': '0.0',
   'pressure': '1011',
   'pressureInches': '30',
   'temp_C': '23',
   'temp_F': '74',
   'uvIndex': '0',
   'visibility': '10',
   'visibilityMiles': '6',
   'weatherCode': '176',
   'weatherDesc': [{'value': 'Patchy rain nearby'}],
   'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0025_light_rain_showers_night.png'}],
   'winddir16Point': 'WSW',
   'winddirDegree': '241',
   'windspeedKmph': '15',
   'windspeedMiles': '9'}],
 'nearest_area': [{'areaName': [{'value': 'Shikinejima'}],
   'country': [{'value': 'Japan'}],
   'latitude': '34.327',
   'longitude': '139.218',
   'population': '0',
   'region': [{'value': 'Tokyo'}],
   'weatherUrl': [{'value': 'https://www.worldweatheronline.com/v2/weather.aspx?q=34.327,139.218'}]}],
 'request': [{'q

In [31]:
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver

@dataclass
class Context:
    user_id: str

@dataclass
class ResponseFormat:
    summary: str
    temperature_celsius: float
    temperature_fahrenheit: float
    humidity: float

@tool
def locate_user(runtime: ToolRuntime[Context]):
    """A tool that locates user's city based on the context"""
    match runtime.context.user_id:
        case 'ABC123':
            return "Vienna"
        case 'XYZ456':
            return "Paris"
        case 'HJKL111':
            return "Tokyo"
        case _:
            return "Unknown"
        

    

In [55]:
from langchain.agents import create_agent


# llm = ChatGroq(
#     model="llama-3.3-70b-versatile", #"qwen/qwen3-32b",
#     temperature=0,
#     #reasoning_format="hidden"
# )
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_weather, locate_user],
    system_prompt="You are a helpful assistant.",
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

# invoke

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather like?"}
        ],
    },
    config={"configurable": {"thread_id": "1"}},
    context=Context(user_id="ABC123")
)

print(response["messages"][-1])

content='{"temperature_celsius":28,"temperature_fahrenheit":83,"humidity":48,"summary":"The current weather in Vienna is clear with a temperature of 28°C (83°F) and humidity at 48%. The wind is blowing from the WSW at 10 km/h (6 mph). No precipitation is observed."}' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 9882, 'total_tokens': 9953, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_67c656fb7a', 'id': 'chatcmpl-DvrTf8Svage2Tdba7wBkIPX2tNUHn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f102a-495f-74d1-8c2f-29520d454fa9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 9882, 'output_tokens': 

In [61]:
print(response["structured_response"])
type(response["structured_response"])

ResponseFormat(summary='The current weather in Vienna is clear with a temperature of 28°C (83°F) and humidity at 48%. The wind is blowing from the WSW at 10 km/h (6 mph). No precipitation is observed.', temperature_celsius=28.0, temperature_fahrenheit=83.0, humidity=48.0)


__main__.ResponseFormat

In [57]:
type(response["messages"][-1].content)

str

In [58]:
import json

json.loads(response["messages"][-1].content)

{'temperature_celsius': 28,
 'temperature_fahrenheit': 83,
 'humidity': 48,
 'summary': 'The current weather in Vienna is clear with a temperature of 28°C (83°F) and humidity at 48%. The wind is blowing from the WSW at 10 km/h (6 mph). No precipitation is observed.'}

### Basic RAG

In [69]:
from langchain_openai import OpenAIEmbeddings
#from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings(model="text-embedding-3-large")
#embedding.embed_documents(["this is an example text"])

texts = [
    (
        "An apple is a crisp, edible fruit produced by the deciduous Malus domestica tree. "
        "Belonging to the rose family (Rosaceae), it is one of the most widely cultivated and "
        "consumed fruits in the world."
    ),
    (
        "A bicycle (often called a bike, push-bike, or cycle) is a human-powered or motor-assisted, "
        "single-track vehicle with two wheels attached to a frame, one behind the other. "
        "Riders sit on a saddle, steer with handlebars, and propel the vehicle by turning two foot-pedals."
    ),
    (
        "GPU stands for Graphics Processing Unit. It is a specialized electronic circuit designed to "
        "rapidly manipulate and alter memory to accelerate the creation of images, 3D graphics, and videos."
    )
]

#vector_store = FAISS.from_texts(texts, embedding=embedding)
vector_store = Chroma.from_texts(texts, embedding=embedding)

vector_store.similarity_search("What is a bicycle?", k=1)

[Document(id='653c0093-a30c-4727-b6ee-79d2dcd04cf1', metadata={}, page_content='A bicycle (often called a bike, push-bike, or cycle) is a human-powered or motor-assisted, single-track vehicle with two wheels attached to a frame, one behind the other. Riders sit on a saddle, steer with handlebars, and propel the vehicle by turning two foot-pedals.')]

In [76]:
# convert to a retriever tool
retriever = vector_store.as_retriever(search_kwargs={'k': 2})

result = retriever.invoke("What is a bicyle")
print(result)

# convert the retriever to a tool
@tool
def retrieve_docs(query: str):
    """A tool that retrieves semantically relevent documents for a given query"""
    result = retriever.invoke(query)
    return result

# similar approach for converting to a tool
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    name="kb_searcher",
    description="Search the knowledge base for information"
)


[Document(id='653c0093-a30c-4727-b6ee-79d2dcd04cf1', metadata={}, page_content='A bicycle (often called a bike, push-bike, or cycle) is a human-powered or motor-assisted, single-track vehicle with two wheels attached to a frame, one behind the other. Riders sit on a saddle, steer with handlebars, and propel the vehicle by turning two foot-pedals.'), Document(id='9f540ac5-d6c3-46ad-9650-b3b05b87ea03', metadata={}, page_content='A bicycle (often called a bike, push-bike, or cycle) is a human-powered or motor-assisted, single-track vehicle with two wheels attached to a frame, one behind the other. Riders sit on a saddle, steer with handlebars, and propel the vehicle by turning two foot-pedals.')]


In [77]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


agent = create_agent(
    model=llm,
    tools=[retrieve_docs],
    system_prompt="You are a hlpful assistant. To answer questions, first search through the knowldge base to retrieve context",
)

result = agent.invoke(
    {
        "messages": ["What is a bicycle?"],
    },
    config={"configurable": {"thread_id": 1}},
    context=Context(user_id="ABC123")
)

In [78]:
result

{'messages': [HumanMessage(content='What is a bicycle?', additional_kwargs={}, response_metadata={}, id='b5ee7589-8040-4efb-975d-926ef0446c20'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 80, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5efe265edd', 'id': 'chatcmpl-DvsVIEP4jdcvql6I1Cf7leHC4bg57', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1066-7e0a-7fa1-a108-32cdbda48317-0', tool_calls=[{'name': 'retrieve_docs', 'args': {'query': 'bicycle definition'}, 'id': 'call_1YKD5LbgTv2kr3I32wlzUoCV', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, '